# DIT Natural Language Processing lesson 2026

## Recurrent neural networks

In this lesson, we build a recurrent neural network (RNN) to treat text as a sequence of words.

In [ ]:
! pip install gensim

In [ ]:
# Importing the dependencies
import glob
import numpy as np
import os
import pandas as pd
# import tarfile

from gensim.models.keyedvectors import KeyedVectors
from keras.models import Sequential
from keras.layers import Dense, Dropout, Input, Flatten, SimpleRNN
from nltk.tokenize import TreebankWordTokenizer
from random import shuffle
from tqdm.auto import tqdm  # to show a smart progress meter
from urllib import request

In [ ]:
PATH_TO_GOOGLENEWS_VECTORS ="https://www.dropbox.com/s/965dir4dje0hfi4/GoogleNews-vectors-negative300.bin.gz?dl=1"
GOOGLE_VECTORS = "GoogleNews-vectors-negative300.bin.gz"

def download_file(url_to_file, path_to_file):
  if os.path.isfile(path_to_file):
    print("A local copy of the file exists already:", path_to_file, "\nDoing nothing")
  else:
    request.urlretrieve(url_to_file, path_to_file)

def download_corpus():
    """
    A function to download and shuffle the data (through huggingface)

    The dataset is at https://huggingface.co/datasets/stanfordnlp/imdb
    """

    splits = {'train': 'plain_text/train-00000-of-00001.parquet',
              'test': 'plain_text/test-00000-of-00001.parquet',
              'unsupervised': 'plain_text/unsupervised-00000-of-00001.parquet'}
    df = pd.read_parquet("hf://datasets/stanfordnlp/imdb/" + splits["train"])

    # shuffle the dataset
    df = df.sample(frac=1).reset_index(drop=True)
    return df

In [ ]:
# Downloading the embeddings and loading the embeddings

download_file(PATH_TO_GOOGLENEWS_VECTORS, GOOGLE_VECTORS)
word_vectors = KeyedVectors.load_word2vec_format(GOOGLE_VECTORS,
    binary=True, limit=400000)

# Downloading and loading the corpus
dataset = download_corpus()
dataset

**Note**: I am using the same methods as in the previous session.
I could simply store them all in a .py file and import them, as
with the libraries

In [ ]:
# Tokenizing and vectorizing all the instances
def tokenize_and_vectorize(dataset):
  tokenizer = TreebankWordTokenizer()
  vectorized_data = []
  # expcted = [] this line appears in the book, but it's not necessary here!
  for sample in tqdm(dataset['text']):
    # print(sample)
    tokens = tokenizer.tokenize(sample)
    sample_vecs = []
    for token in tokens:
      try:
        sample_vecs.append(word_vectors[token])
      except KeyError:
        pass # No matching token in the Google w2v vocab
    np_vec = np.vstack(sample_vecs)
    vectorized_data.append(np_vec)
  return vectorized_data

In [ ]:
# Extracting the expected output for all the instances
def collect_expected(dataset):
    """Peel off the target values (classes) from the dataset """
    expected = []
    for sample in dataset['label']:
        expected.append(sample)
    return expected

In [ ]:
# Vectorising the dataset and extracting the gold standard
x = tokenize_and_vectorize(dataset)
y = collect_expected(dataset)

In [ ]:
print(x[0])

In [ ]:
# Define training and validation data (even if it's called test here)
split_point = int(len(x)*.8)

x_train = x[:split_point] # there's a typo in this line, if copying from the book
y_train = y[:split_point]
# This is actually a validation partition. I keep the names from the book
x_test = x[split_point:]
y_test = y[split_point:]

In [ ]:
# Network parameters
maxlen = 400
batch_size = 32
embedding_dims = 300
epochs = 2
num_neurons = 50

In [ ]:
# Padding and truncating the input is not strictly necessary for RNNs;
# we are applying here to have a fair comparison against the CNN (same
# input length)
class Collator:
  def __init__(self,
                maxlen,
                batch_size,
                ) -> None:
    self.maxlen = maxlen
    self.batch_size = batch_size

  def padding_and_truncating(self, x, y):
    """
    Add zeros at the end of the representation for short instances,
    truncate longer ones to the maxlen
    """
    vec_dim = len(x[0][0])
    N = len(x)
    X = np.zeros((N, self.maxlen, vec_dim))  # preallocate a np array
    Y = np.array(y)

    for i, tokens in enumerate(x):
      length = min(len(tokens), self.maxlen)
      if length > 0:
        X[i, :length] = np.asarray(tokens[:length])  # fill the np array
    return X, Y

  def collate(self, X, Y, N, epochs = 1):
    """
    This method is used to feed batches into the `model.fit` method
    """
    for _ in range(epochs):
      """
      This `for _ in range(epochs):` loop is here because
      the `for` below needs to be able to be called `epochs` times
      so each time the `for` is called the iterator is replenished
      for the new epoch
      """
      for start in range(0, N, self.batch_size):
        end = start + self.batch_size
        x_batch, y_batch = self.padding_and_truncating(
          X[start:end],
          Y[start:end]
          )
        yield x_batch, y_batch

In [ ]:
collator = Collator(maxlen = maxlen, batch_size = batch_size)

In [ ]:
# Initializing the (empty) network
model = Sequential()

In [ ]:
# Adding one recurrent layer

# In previous versions of keras (and in the book), the input
# shape was defined as an argument to SimpleRNN. That way
# still works, but adding an Input instead is adviced
model.add(Input([maxlen, embedding_dims]))
model.add(SimpleRNN(
    num_neurons,
    return_sequences=True,
    # input_shape=(maxlen, embedding_dims)
    )
  )

Homework: experiment with return_sequences=False and compare the results

**Back to the slides**

In [ ]:
# Adding a dropout layer (remember why?)
model.add(Dropout(.2))

# Adding a flattening layer
model.add(Flatten())

# Adding the classifier
model.add(Dense(1, activation='sigmoid'))

**Flatten?** back to the slides

In [ ]:
# Compiling the network
model.compile('rmsprop',
              'binary_crossentropy',
              metrics=['accuracy'])
model.summary()

# 37,551 parameters!

In [ ]:
# Training the network

# model.fit(x_train, y_train,
#     batch_size=batch_size,
#     epochs=epochs,
#     validation_data=(x_test, y_test))

model.fit(
    collator.collate(x_train, y_train, N = len(x_train), epochs=epochs),
    steps_per_epoch=len(x_train) // batch_size,
    validation_data=collator.collate(x_test, y_test, N = len(x_test), epochs=epochs),
    validation_steps=len(x_test) // batch_size,
    epochs=epochs,
    )

**Back to the slides**

In [ ]:
# Building a bigger network
num_neurons = 100
model = Sequential()
model.add(SimpleRNN(
    num_neurons,
    return_sequences=True,
    input_shape=(maxlen, embedding_dims))
     )
model.add(Dropout(.2))
model.add(Flatten())
model.add(Dense(1, activation='sigmoid'))
model.compile('rmsprop', 'binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# Training
# model.fit(x_train,
#     y_train,
#     batch_size=batch_size,
#     epochs=epochs,
#     validation_data=(x_test, y_test)
#      )
model.fit(
    collator.collate(x_train, y_train, N = len(x_train), epochs=epochs),
    steps_per_epoch=len(x_train) // batch_size,
    validation_data=collator.collate(x_test, y_test, N = len(x_test), epochs=epochs),
    validation_steps=len(x_test) // batch_size,
    epochs=epochs,
    )

The improvement is tiny*$\rightarrow$ perhaps the network is too complex.

\* (depending on the random initialisation, it could be even worst!)

Homework: try with 25 neurons

In [ ]:
# Saving the network
model_structure = model.to_json()
with open("simplernn_model2.json", "w") as json_file:
    json_file.write(model_structure)
model.save_weights("simplernn2.weights.h5")

Predicting on new instances

In [ ]:
# Notice we have both positive and negative words here
sample_negative = """I hate that the dismal weather had me down for so long,
when will it break! Ugh, when does happiness return? The sun is
blinding and the puffy clouds are too thin. I can't wait for the weekend."""
# Super positive sample
sample_positive = """I love that incredible weather!
I feel like this happiness will last forever!
The sun is super nice and the breeze is soothing.
I could stay like this forever.
"""

# The first value is a "fake" class (this is the expected input)
new_instances = [
    {"text": sample_negative, "label": 0},
    {"text": sample_positive, "label": 1}
    ]
data_dummy = pd.DataFrame.from_records(new_instances)

x_dummy = tokenize_and_vectorize(data_dummy)
y_dummy = collect_expected(data_dummy)

x_dummy_padded, y_dummy = collator.padding_and_truncating(x_dummy, y_dummy)
preds = model.predict(x_dummy_padded)
print(preds)

In [ ]:
# Get the class by thresholding the logits
(preds > 0.5).astype("int32")

**End of the notebook**